<a href="https://colab.research.google.com/github/closes/earth_observation_samos/blob/main/Working_with_SST_and_SSH_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Option 1: download the data within python.

For these exercises, if you want to download the data directly within python, then you will start by installing the `copernicusmarine` toolbox again so that you can access the data:

In [ ]:
!pip install copernicusmarine

In [ ]:
import copernicusmarine as cm

And connect to your account:

In [ ]:
cm.login(username="enter_your_username")

# Option 2: download the data from the Copernicus Marine website

If you prefer to download the data using the graphical interface or the subsetting tool, then you can download the data directly from the [Copernicus Marine website](https://marine.copernicus.eu/), and then load it into python afterwards.

# A. Tracking fronts in SST data

## Loading the data

We'll use the [ESA SST CCI and C3S reprocessed sea surface temperature analyses](https://data.marine.copernicus.eu/product/SST_GLO_SST_L4_REP_OBSERVATIONS_010_024/description) product, which has product ID: SST_GLO_SST_L4_REP_OBSERVATIONS_010_024. The spatial resolution is 0.05°x0.05°. Find this product in the Copernicus Marine Data catalogue.



We will work with the `sea surface temperature` variable from the dataset called "Copernicus Climate Change Service".

We will load the SST data for the region 15-30°E, 30-40°S on the 5th September 2023.

If you prefer to download the data from the Copernicus Marine website directly, you should do that now and skip the following cell.

If you want to load the data directly into python, you can fill in your code in the cell below:

In [ ]:
ds = cm.open_dataset(
    dataset_id= ,# string
    variables= , # list of strings
    start_datetime= , #string
    end_datetime= , #string
    minimum_longitude= , # float
    maximum_longitude= , # float
    minimum_latitude= , # float
    maximum_latitude= , # float
)

In [ ]:
ds

Next steps:
1. Inspect your data set: how many time steps are there, and how many longitude and latitude values do you have?
2. Plot the SST as a map. Where do you think the strongest fronts might be on your map?

In [ ]:
# enter your code here

### Front detection: calculating the sea surface temperature gradient

Now, let's calculate the gradient of the Sea Surface Temperature (SST) field, denoted as `grad(T)`. The gradient tells us the rate of change of temperature with respect to spatial dimensions (longitude and latitude).

Mathematically, the gradient of a scalar field $T$ (like temperature) in 2D (longitude, latitude) can be expressed as:

$$ \nabla T = \left( \frac{\partial T}{\partial \text{longitude}}, \frac{\partial T}{\partial \text{latitude}} \right) $$

In simple terms, we are calculating how steeply the temperature changes when moving a small distance in the longitude direction (`dT/dLon`) and how steeply it changes when moving a small distance in the latitude direction (`dT/dLat`). The magnitude of this vector will show us where the temperature changes are most significant (e.g., at oceanic fronts).

We can calculate gradients using numpy's gradient function. This will give us dT: the difference between consecutive values of T. We need to choose which direction to calculate it in: axis=0 will correspond to latitude and axis=1 will correspond to longitude.

We can obtain d(longitude) and d(latitude) by inspecting our latitude and longitude data. The grid is regular, and we know that the data set spatial resolution is 0.05°x0.05°. We can confirm this by subtracting two consecutive latitude and longitude values:

In [ ]:
# Since we only have one time step in our outputs, let's get rid of the
# time dimension. This will mean that we no longer have to select the
# time step each whenever we want to access the data
ds = ds.isel(time=0)
ds

In [ ]:
# Calculate dT in the longitude direction using np.gradient
dT_lon = np.gradient(ds['analysed_sst'].values, axis=1)
# print the first 2 longitude values for the data set
print('First 2 longitude values: ', ds['longitude'].values[:2])
# calculate dlon as the difference between two successive values:
lon = ds['longitude'].values
dlon = lon[1] - lon[0]
print('Difference between longitude values: ', dlon)

So finally we can calculate `dT/dLon` by dividing the `dT_lon` term by 0.05.

In the cell below, calculate both dT/d(longitude) and dT/d(latitude) for the time step selected above.

You should have two arrays at the end, and each one should be the same size as the original `analysed_sst` array.

In [ ]:
# enter your code here

We can now calculate the amplitude of $\nabla T$ to characterise the gradient. This is defined as:
$$|\nabla T|=\sqrt{\left (\frac{\partial T}{\partial\, longitude}\right )^2 + \left (\frac{\partial T}{\partial\, latitude}\right )^2}$$
Where you have calculated:
- `dT/dLon` -> $\frac{\partial T}{\partial\, longitude}$
- `dT/dLat` -> $\frac{\partial T}{\partial\, latitude}$

in the previous step.

Calculate $|\nabla T|$ using these terms below and then plot your output, $|\nabla T|$, using `matplotlib`'s `pcolormesh` function.

In [ ]:
# enter your code here

Finally, to determine the front positions, we will apply a threshold to $|\nabla T|$. All values of $|\nabla T|$ that exceed this value will be considered to indicate the presence of fronts.

You will not know in advance what a good choice of value for the threshold is. To help to determine this, you can plot your array of $|\nabla T|$ values again, using `pcolormesh`, and fix the upper colour limit, so that some of the values exceed this. You can do this by setting `vmax`, in the plotting command. For example, if you want to set the threshold at 14 and your array is called `dT`, you would write plt.pcolormesh(dT, vmax=14).

Do some tests in the cell below until you find a value of vmax that highlights the strong gradients well.

In [ ]:
# enter your code here

You can now create a second array that contains a value of 1 at the positions where your $|\nabla T|$ array exceeds the threshold, and 0 at all other positions. This will indicate which pixels you consider to correspond to oceanic fronts.

You can perform this calculation by applying a boolean condition in python. For example, if we have an array A, and we want to create an array B of the same size that contains either 0 or 1 depending on the value of A:


*   1 if A < 5
*   0 for all other values of A

we can write:
`B = A < 5` to create a boolean (True/False) array. This can then be turned into values or 0 (=False) and 1 (=True) by transforming the array values to integers.

Let's see it work for this example:

In [ ]:
A = np.array([1, 2, 5, 6, 7])
print('A=',A)
B = A < 5
print('B=',B)
print('And converting to integer: B=',B.astype(int))

You can apply this technique to your array of values of $|\nabla T|$ to create a second array that will indicate the positions of the fronts.

Instead of 5, as in the example above, you will use the threshold value of $|\nabla T|$ that you chose based on your plots above to create your second array.

Where $|\nabla T|$ is less than the threshold value, your second array will contain the value `False`, and where $|\nabla T|$ is greater than the threshold value, your second array will contain the value `True`.

In [ ]:
# enter your code here to create the threshold array

Now make a final plot showing:

*   a pcolormesh plot of the original SST data
*   on the same figure: a contour plot, showing the data from your threshold array



In [ ]:
# enter your code here

The lines of the threshold contour plot should correspond to the regions where there are rapid changes in temperature. How well did the method work?

Here we have calculated the gradient as: $$ \nabla T = \left( \frac{\partial T}{\partial \text{longitude}}, \frac{\partial T}{\partial \text{latitude}} \right) $$

What are the units of $\nabla T$? Are there any disadvantages to these units? And how might you convert the units to °C/m?

## B. Detecting ocean eddies using sea level data and velocities

We'll use the ["Global Ocean Gridded L4 Sea Surface Heights And Derived Variables Reprocessed Copernicus Climate Service"](https://data.marine.copernicus.eu/product/SEALEVEL_GLO_PHY_CLIMATE_L4_MY_008_057/description) product, which has product ID: SEALEVEL_GLO_PHY_CLIMATE_L4_MY_008_057. The spatial resolution is 0.25°x0.25°. Find this product in the Copernicus Marine Data catalogue.

We'll download the same time step as for part A, but over a larger region:
- 5°W-40°E, 30-45°S on the 5th September 2023.

You will need to download 3 variables:
1. Sea surface height above sea level
2. Surface geostrophic eastward sea water velocity assuming sea level for geoid
3. Surface geostrophic northward sea water velocity assuming sea level for geoid

These correspond to the sea level anomaly (SLA) and geostrophic velocity anomalies that we saw in lecture 2.

If you want to download the data directly from the Copernicus Marine website you should do that now.

If you want to load the data directly into python, then you can fill in your code in the cell below:

In [ ]:
ds = cm.open_dataset(
    dataset_id= ,# string
    variables= , # list of strings
    start_datetime= , #string
    end_datetime= , #string
    minimum_longitude= , # float
    maximum_longitude= , # float
    minimum_latitude= , # float
    maximum_latitude= , # float
)

Let's inspect the data before continuing:

In [ ]:
ds

The aim of this exercise will be to try to detect eddies automatically in the altimetry data.

We can do this using a quantity called the *Okubo-Weiss parameter*.

The Okubo-Weiss parameter is constructed from several measures of how the water is moving. The key idea is to distinguish between two different types of motion:

- *Rotation*: water moves around a common centre, as it does in an eddy.
- *Strain*: water is stretched and compressed as neighbouring parcels of water move apart in one direction and together in another.

We can calculate quantities that describe the strength of these two types of motion. The Okubo-Weiss parameter compares them: where rotation is stronger than stretching, the cores of eddies may be present.

Imagine releasing a small group of dye particles into the ocean.

- If the particles start moving around one another, the water is rotating.

- If the particles are pulled apart and stretched into a long filament, the water is undergoing strain.

An eddy is a region where the first behaviour dominates.

To make the eddy detection work better, we will slightly smooth the data before we use it. Before continuing, run the cell below, which will average over 1.25° areas to remove small scale variability. We will then remove the edge pixels from the data set, which are not well defined after we apply the smoothing:

In [ ]:
# average over the 1.25° window (=5 grid cells)
ds = ds.isel(time=0).rolling(
    latitude=5,
    longitude=5,
    center=True
).mean()

# trim the data set to remove the edge pixels
ds = ds.isel(
    latitude=slice(2, -2),
    longitude=slice(2, -2)
)
# print out dataset summary
ds

Note that we also got rid of the time dimension at the same time.

Now make a plot of the sea surface height data. Can you see any features that look like eddies?

In [ ]:
# enter your code here

Mathematically, the Okubo-Weiss parameter is defined as:
$$W = s_n^2 + s_s^2 - \zeta^2$$
where:
- $\zeta$ is the relative velocity, and represents rotation
- $s_n$ is the normal strain, and represents stretching or compression
- $s_s$ is the shear strain, and represents shear in the fluid

We'll see how to calculate these terms below.

The value of W tells us whether we expect to find an eddy at each grid cell:
- W < 0: rotation dominates, an eddy may be present
- W > 0: strain domainates, no eddy expected to be present


### Calculation of the Okubo-Weiss parameter

The different terms $s_n$, $s_s$ and $\zeta$ are all formed from different combinations of velocity gradients. We will need to calculate these in a similar way to the approach that you used for SST in the first part of the practical.

The terms are defined as follows:
- Normal strain: $s_n = \frac{\partial u}{\partial x}-\frac{\partial v}{\partial y}$
- Shear strain: $s_s = \frac{\partial u}{\partial y}+\frac{\partial v}{\partial x}$
- Relative vorticity: $\zeta = \frac{\partial v}{\partial x}-\frac{\partial u}{\partial y}$

So we will need the 4 following velocity gradient terms in order to calculate all 3 contributions:
$$\frac{\partial u}{\partial x}, \frac{\partial u}{\partial y}, \frac{\partial v}{\partial x}, \frac{\partial v}{\partial y}$$

In the cell below, use np.gradient to calculate `du` with respect to the longitude and latitude directions.

For this calculation, we must be more precise than for the SST gradient: we need to use dx and dy in metres, so we cannot simply use the grid spacing in degrees as we did last time.

The cell below contains pre-calculated values of dx and dy for your data set. You can then use these values in combination with your gradients to estimate the 4 velocity gradient terms.

In [ ]:
dx = 22055.2  # m
dy = 27800.0  # m
# enter your code here to calculate the gradients

Now, use those 4 velocity gradient terms to calculate $s_n$, $s_s$ and $\zeta$. Then calculate the Okuko-Weiss parameter, $W$.

In [ ]:
# enter your code here

Finally, apply a threshold, like you did for the SST exercise, to create a second array where:
- points where W>0 take value 0 in the new array
- points where W<0 take value 1 in the new array

Plot the threshold array using pcolormesh.

In [ ]:
# enter your code here

You will probably find that the criterion W<0 is not strict enough and that the criterion detects eddies almost everywhere!

Below, produce a figure showing the SLA with the Okubo-Weiss threshold array plotted over the top as contours.

In [ ]:
# enter your code here

Now let's try to change the value at which we apply the threshold.

Like for the SST example, test different threshold values until you find one that works well for the eddy detection. You might find it helpful to define this number using the standard deviation of W, for example:

```
sigma_W = np.nanstd(W)
threshold = -0.1 * sigma_W
```
You can change the multiplying factor (0.1) to see how this changes your results.

As before, plot SLA and the threshold array on the same figure to see where you have detected eddies in the original array.

In [ ]:
# enter your code here